In [25]:
import os
import json
import shutil
from pathlib import Path
from PIL import Image
import numpy as np

In [26]:
def main():
    print("=" * 60)
    print("  🌿 Plant Disease Detector – Dataset Setup")
    print("=" * 60)

    out_dir = "dataset/PlantVillageDataset/PlantVillage"

    if os.path.isdir(out_dir) and len(os.listdir(out_dir)) > 5:
        print(f"\n✅ Dataset already present at '{out_dir}'. Nothing to do.")
        classes = os.listdir(out_dir)
        print(f"   Found {len(classes)} class folders.")
        return

    ok = download_via_tfds(out_dir)
    if not ok:
        print(KAGGLE_INSTRUCTIONS)

if __name__ == "__main__":
    main()


  🌿 Plant Disease Detector – Dataset Setup

✅ Dataset already present at 'dataset/PlantVillageDataset/PlantVillage'. Nothing to do.
   Found 15 class folders.


In [27]:
import os
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

In [28]:
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator


In [29]:
IMG_SIZE     = 224          # MobileNetV2 default
BATCH_SIZE   = 32
EPOCHS       = 30
LR           = 1e-4
DATASET_DIR  = "dataset/PlantVillageDataset/PlantVillage"   # folder created by download_dataset.py
MODEL_DIR    = "models"
LOGS_DIR     = "logs"

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(LOGS_DIR,  exist_ok=True)

In [30]:
def build_generators(dataset_dir: str):
    train_gen = ImageDataGenerator(
        rescale=1./255,
        rotation_range=40,
        width_shift_range=0.2,
        height_shift_range=0.2,
        shear_range=0.2,
        zoom_range=0.2,
        horizontal_flip=True,
        vertical_flip=True,
        brightness_range=[0.8, 1.2],
        validation_split=0.2,
    )
    val_gen = ImageDataGenerator(
        rescale=1./255,
        validation_split=0.2,
    )

    train_ds = train_gen.flow_from_directory(
        dataset_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        subset="training",
        shuffle=True,
        seed=42,
    )
    val_ds = val_gen.flow_from_directory(
        dataset_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode="categorical",
        subset="validation",
        shuffle=False,
        seed=42,
    )
    return train_ds, val_ds



In [31]:
# MODEL  (MobileNetV2 + custom head)
# ──────────────────────────────────────────────
def build_model(num_classes: int) -> Model:
    base = MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
    )
    # Freeze base initially
    base.trainable = False

    x = base.output
    x = GlobalAveragePooling2D()(x)
    x = BatchNormalization()(x)
    x = Dense(256, activation="relu")(x)
    x = Dropout(0.5)(x)
    x = Dense(128, activation="relu")(x)
    x = Dropout(0.3)(x)
    out = Dense(num_classes, activation="softmax")(x)

    model = Model(inputs=base.input, outputs=out)
    model.compile(
        optimizer=Adam(learning_rate=LR),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model, base


def unfreeze_top_layers(model, base_model, num_layers: int = 30):
    """Fine-tune the top N layers of the base model."""
    base_model.trainable = True
    for layer in base_model.layers[:-num_layers]:
        layer.trainable = False
    model.compile(
        optimizer=Adam(learning_rate=LR / 10),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    print(f"Fine-tuning: unfroze last {num_layers} layers of MobileNetV2.")



In [32]:
# CALLBACKS
# ──────────────────────────────────────────────
def get_callbacks():
    return [
        ModelCheckpoint(
            filepath=os.path.join(MODEL_DIR, "best_model.keras"),
            monitor="val_accuracy",
            save_best_only=True,
            verbose=1,
        ),
        EarlyStopping(
            monitor="val_accuracy",
            patience=8,
            restore_best_weights=True,
            verbose=1,
        ),
        ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.5,
            patience=4,
            min_lr=1e-7,
            verbose=1,
        ),
        CSVLogger(os.path.join(LOGS_DIR, "training_log.csv")),
    ]


In [33]:
def plot_history(history, filename="training_curves.png"):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle("Plant Disease Model – Training History", fontsize=14, fontweight="bold")

    axes[0].plot(history.history["accuracy"],     label="Train Acc", linewidth=2)
    axes[0].plot(history.history["val_accuracy"], label="Val Acc",   linewidth=2)
    axes[0].set_title("Accuracy"); axes[0].set_xlabel("Epoch"); axes[0].legend()
    axes[0].grid(alpha=0.3)

    axes[1].plot(history.history["loss"],     label="Train Loss", linewidth=2)
    axes[1].plot(history.history["val_loss"], label="Val Loss",   linewidth=2)
    axes[1].set_title("Loss"); axes[1].set_xlabel("Epoch"); axes[1].legend()
    axes[1].grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(LOGS_DIR, filename), dpi=150)
    print(f"✅ Training curves saved → {LOGS_DIR}/{filename}")



In [34]:
# MAIN
# ──────────────────────────────────────────────
def main():
    print("=" * 60)
    print("  🌿 Plant Disease Detector – Training")
    print("=" * 60)

    if not os.path.isdir(DATASET_DIR):
        print(f"\n❌  Dataset not found at '{DATASET_DIR}'")
        print("    Run  python download_dataset.py  first.\n")
        return

    # ── Data ──
    train_ds, val_ds = build_generators(DATASET_DIR)
    num_classes = len(train_ds.class_indices)
    print(f"\n📂 Classes found : {num_classes}")
    print(f"   Train samples : {train_ds.samples}")
    print(f"   Val   samples : {val_ds.samples}\n")

    # Save class mapping
    class_map = {v: k for k, v in train_ds.class_indices.items()}
    with open(os.path.join(MODEL_DIR, "class_names.json"), "w") as f:
        json.dump(class_map, f, indent=2)
    print(f"✅ Class names saved → {MODEL_DIR}/class_names.json")

    # ── Phase 1: Train head only ──
    print("\n── Phase 1: Training head (base frozen) ──")
    model, base_model = build_model(num_classes)
    model.summary(line_length=80)

    history1 = model.fit(
        train_ds,
        epochs=15,
        validation_data=val_ds,
        callbacks=get_callbacks(),
    )

    # ── Phase 2: Fine-tune top layers ──
    print("\n── Phase 2: Fine-tuning top 30 layers ──")
    unfreeze_top_layers(model, base_model, num_layers=30)

    history2 = model.fit(
        train_ds,
        epochs=EPOCHS,
        initial_epoch=len(history1.history["loss"]),
        validation_data=val_ds,
        callbacks=get_callbacks(),
    )

    # Merge histories
    combined = {}
    for k in history1.history:
        combined[k] = history1.history[k] + history2.history[k]

    class FakeHistory:
        def __init__(self, h): self.history = h
    plot_history(FakeHistory(combined))

    # ── Evaluate ──
    print("\n── Final Evaluation on Validation Set ──")
    loss, acc = model.evaluate(val_ds, verbose=1)
    print(f"\n🏆 Val Accuracy : {acc * 100:.2f}%")
    print(f"   Val Loss     : {loss:.4f}")

    # Save final model
    final_path = os.path.join(MODEL_DIR, "plant_disease_model.keras")
    model.save(final_path)
    print(f"\n✅ Final model saved → {final_path}")
    print("\nDone! Run  python predict.py  to classify a new image.\n")


In [35]:
if __name__ == "__main__":
    main()


  🌿 Plant Disease Detector – Training
Found 16516 images belonging to 15 classes.
Found 4122 images belonging to 15 classes.

📂 Classes found : 15
   Train samples : 16516
   Val   samples : 4122

✅ Class names saved → models/class_names.json

── Phase 1: Training head (base frozen) ──


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)          ┃ Output Shape      ┃     Param # ┃ Connected to       ┃
┡━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1         │ (None, 224, 224,  │           0 │ -                  │
│ (InputLayer)          │ 3)                │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ Conv1 (Conv2D)        │ (None, 112, 112,  │         864 │ input_layer_1[0][… │
│                       │ 32)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ bn_Conv1              │ (None, 112, 112,  │         128 │ Conv1[0][0]        │
│ (BatchNormalization)  │ 32)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ Conv1_relu (ReLU)     │ (None, 112, 112,  │           0 │ bn_Conv1[0][0]     │
│                       │ 32)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ expanded_conv_depthw… │ (None, 112, 112,  │         288 │ Conv1_relu[0][0]   │
│ (DepthwiseConv2D)     │ 32)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ expanded_conv_depthw… │ (None, 112, 112,  │         128 │ expanded_conv_dep… │
│ (BatchNormalization)  │ 32)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ expanded_conv_depthw… │ (None, 112, 112,  │           0 │ expanded_conv_dep… │
│ (ReLU)                │ 32)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ expanded_conv_project │ (None, 112, 112,  │         512 │ expanded_conv_dep… │
│ (Conv2D)              │ 16)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ expanded_conv_projec… │ (None, 112, 112,  │          64 │ expanded_conv_pro… │
│ (BatchNormalization)  │ 16)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ block_1_expand        │ (None, 112, 112,  │       1,536 │ expanded_conv_pro… │
│ (Conv2D)              │ 96)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ block_1_expand_BN     │ (None, 112, 112,  │         384 │ block_1_expand[0]… │
│ (BatchNormalization)  │ 96)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ block_1_expand_relu   │ (None, 112, 112,  │           0 │ block_1_expand_BN… │
│ (ReLU)                │ 96)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ block_1_pad           │ (None, 113, 113,  │           0 │ block_1_expand_re… │
│ (ZeroPadding2D)       │ 96)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ block_1_depthwise     │ (None, 56, 56,    │         864 │ block_1_pad[0][0]  │
│ (DepthwiseConv2D)     │ 96)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ block_1_depthwise_BN  │ (None, 56, 56,    │         384 │ block_1_depthwise… │
│ (BatchNormalization)  │ 96)               │             │                    │
├───────────────────────┼───────────────────┼─────────────┼────────────────────┤
│ block_1_depthwise_re… │ (None, 56, 56,    │           0 │ block_1_depthwise… │
│ (ReLU)                │ 96)  

 Total params: 2,625,871 (10.02 MB)

 Trainable params: 365,327 (1.39 MB)

 Non-trainable params: 2,260,544 (8.62 MB)

Epoch 1/15
517/517 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.2487 - loss: 2.6664
Epoch 1: val_accuracy improved from None to 0.65332, saving model to models\best_model.keras

Epoch 1: finished saving model to models\best_model.keras
517/517 ━━━━━━━━━━━━━━━━━━━━ 803s 2s/step - accuracy: 0.3712 - loss: 2.0974 - val_accuracy: 0.6533 - val_loss: 1.1413 - learning_rate: 1.0000e-04
Epoch 2/15
517/517 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.5408 - loss: 1.4323
Epoch 2: val_accuracy improved from 0.65332 to 0.70645, saving model to models\best_model.keras

Epoch 2: finished saving model to models\best_model.keras
517/517 ━━━━━━━━━━━━━━━━━━━━ 666s 1s/step - accuracy: 0.5694 - loss: 1.3340 - val_accuracy: 0.7065 - val_loss: 0.8841 - learning_rate: 1.0000e-04
Epoch 3/15
517/517 ━━━━━━━━━━━━━━━━━━━━ 0s 1s/step - accuracy: 0.6365 - loss: 1.1314
Epoch 3: val_accuracy improved from 0.70645 to 0.75328, saving model to models\best_model.keras

Epoch 3: finished saving model to models\best_mod

D:\PlantDIsease\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\acer\AppData\Local\Temp\ipykernel_22296\1109999194.py:438: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: css. Please pass these parameters to launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="🌿 Plant Disease Detector") as demo:


* Running on local URL:  http://0.0.0.0:7877
* To create a public link, set `share=True` in `launch()`.


D:\PlantDIsease\venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


In [26]:
"""
Plant Disease Detector – Web App (v5 - theme independent)
Run:  python app.py
"""

import os
import json
import numpy as np
from PIL import Image
import tensorflow as tf
import warnings
warnings.filterwarnings("ignore")

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────
IMG_SIZE             = 224
MODEL_PATH           = "models/plant_disease_model.keras"
BEST_MODEL           = "models/best_model.keras"
CLASS_MAP_PATH       = "models/class_names.json"
CONFIDENCE_THRESHOLD = 70.0

def load_resources():
    path = MODEL_PATH if os.path.exists(MODEL_PATH) else BEST_MODEL
    if not os.path.exists(path):
        raise FileNotFoundError("No trained model found. Run python train.py first.")
    model = tf.keras.models.load_model(path)
    with open(CLASS_MAP_PATH) as f:
        raw = json.load(f)
    return model, {int(k): v for k, v in raw.items()}

try:
    MODEL, CLASS_MAP = load_resources()
    MODEL_READY = True
    LOAD_ERROR  = ""
except Exception as e:
    MODEL_READY = False
    LOAD_ERROR  = str(e)

# ──────────────────────────────────────────────
# Disease database
# ──────────────────────────────────────────────
DISEASE_DB = {
    "healthy": {
        "display":   "Healthy Plant",
        "emoji":     "✅",
        "severity":  "None",
        "border":    "#4caf50",
        "bar":       "#4caf50",
        "remedy":    "Your plant looks healthy! Continue regular watering, ensure adequate sunlight, and inspect leaves weekly for early stress signs.",
        "learn_url": "https://extension.umn.edu/plant-health",
        "prev_url":  "https://www.rhs.org.uk/prevention-protection/plant-health",
    },
    "Early_blight": {
        "display":   "Early Blight",
        "emoji":     "🍂",
        "severity":  "Moderate",
        "border":    "#ff9800",
        "bar":       "#ff9800",
        "remedy":    "Remove and destroy infected leaves immediately. Apply copper-based or chlorothalonil fungicide every 7-10 days. Avoid wetting foliage when watering.",
        "learn_url": "https://extension.umn.edu/disease-management/early-blight-tomato-and-potato",
        "prev_url":  "https://www.gardeningknowhow.com/edible/vegetables/tomato/tomato-early-blight-prevention.htm",
    },
    "Late_blight": {
        "display":   "Late Blight",
        "emoji":     "⚠️",
        "severity":  "High",
        "border":    "#f44336",
        "bar":       "#f44336",
        "remedy":    "Act immediately — Late Blight spreads rapidly. Remove and bag all infected material. Apply mancozeb or a systemic fungicide. Destroy severely infected plants.",
        "learn_url": "https://extension.umn.edu/disease-management/late-blight",
        "prev_url":  "https://www.rhs.org.uk/disease/potato-blight",
    },
    "Leaf_Mold": {
        "display":   "Leaf Mold",
        "emoji":     "🟤",
        "severity":  "Moderate",
        "border":    "#ff9800",
        "bar":       "#ff9800",
        "remedy":    "Increase air circulation by pruning dense foliage. Apply copper-based or mancozeb fungicide. Keep humidity below 85% in greenhouses.",
        "learn_url": "https://www.canr.msu.edu/news/tomato_leaf_mold_in_high_tunnels_and_greenhouses",
        "prev_url":  "https://extension.umn.edu/disease-management/leaf-mold-tomato",
    },
    "Septoria_leaf_spot": {
        "display":   "Septoria Leaf Spot",
        "emoji":     "🟠",
        "severity":  "Moderate",
        "border":    "#ff9800",
        "bar":       "#ff9800",
        "remedy":    "Remove lower infected leaves. Apply chlorothalonil or copper fungicide at first sign. Avoid overhead irrigation. Rotate crops next season.",
        "learn_url": "https://extension.umn.edu/disease-management/septoria-leaf-spot-tomato",
        "prev_url":  "https://www.almanac.com/pest/septoria-leaf-spot",
    },
    "Spider_mites": {
        "display":   "Spider Mites",
        "emoji":     "🕷️",
        "severity":  "Moderate",
        "border":    "#ff9800",
        "bar":       "#ff9800",
        "remedy":    "Spray plants with a strong water jet to dislodge mites. Apply neem oil or insecticidal soap every 5-7 days. Increase ambient humidity.",
        "learn_url": "https://extension.umn.edu/yard-and-garden-insects/spider-mites",
        "prev_url":  "https://www.almanac.com/pest/spider-mites",
    },
    "Target_Spot": {
        "display":   "Target Spot",
        "emoji":     "🎯",
        "severity":  "Moderate",
        "border":    "#ff9800",
        "bar":       "#ff9800",
        "remedy":    "Apply azoxystrobin or pyraclostrobin fungicide at first signs. Remove plant debris after harvest. Improve drainage and air circulation.",
        "learn_url": "https://www.plantix.net/en/library/plant-diseases/200044/target-spot-of-tomato",
        "prev_url":  "https://plantvillage.psu.edu/topics/tomato/infos",
    },
    "Mosaic_virus": {
        "display":   "Tomato Mosaic Virus",
        "emoji":     "🟡",
        "severity":  "High",
        "border":    "#f44336",
        "bar":       "#f44336",
        "remedy":    "No chemical cure exists. Remove and destroy infected plants immediately. Disinfect tools with bleach. Control aphid and whitefly populations.",
        "learn_url": "https://extension.umn.edu/disease-management/tomato-mosaic-virus",
        "prev_url":  "https://www.rhs.org.uk/disease/tomato-mosaic-virus",
    },
    "Yellow_Leaf_Curl_Virus": {
        "display":   "Yellow Leaf Curl Virus",
        "emoji":     "🌀",
        "severity":  "High",
        "border":    "#f44336",
        "bar":       "#f44336",
        "remedy":    "Remove infected plants promptly. Apply imidacloprid or spinosad to control whiteflies. Use reflective mulches to deter insects.",
        "learn_url": "https://plantvillage.psu.edu/topics/tomato/infos",
        "prev_url":  "https://www.plantix.net/en/library/plant-diseases/200028/tomato-yellow-leaf-curl-virus",
    },
    "Bacterial_spot": {
        "display":   "Bacterial Spot",
        "emoji":     "🦠",
        "severity":  "Moderate",
        "border":    "#ff9800",
        "bar":       "#ff9800",
        "remedy":    "Apply copper-based bactericide preventively. Avoid working with wet plants. Remove heavily infected leaves. Use disease-free seed next season.",
        "learn_url": "https://extension.umn.edu/disease-management/bacterial-spot-pepper-and-tomato",
        "prev_url":  "https://www.almanac.com/pest/bacterial-spot",
    },
}

def get_info(class_name):
    for key, info in DISEASE_DB.items():
        if key.lower() in class_name.lower():
            return info
    return {
        "display": class_name.replace("___", " - ").replace("_", " "),
        "emoji": "🔬", "severity": "Unknown",
        "border": "#888", "bar": "#888",
        "remedy": "Consult a local agricultural extension officer for accurate diagnosis.",
        "learn_url": "https://plantvillage.psu.edu/",
        "prev_url":  "https://www.fao.org/plant-health-2020/en/",
    }

# ──────────────────────────────────────────────
# Prediction
# ──────────────────────────────────────────────
def predict(pil_image):
    if pil_image is None:
        return card_waiting(), ""
    if not MODEL_READY:
        return card_error(LOAD_ERROR), ""
    img   = pil_image.convert("RGB").resize((IMG_SIZE, IMG_SIZE))
    arr   = np.array(img, dtype=np.float32) / 255.0
    preds = MODEL.predict(np.expand_dims(arr, 0), verbose=0)[0]
    top   = int(np.argmax(preds))
    conf  = float(preds[top]) * 100
    if conf < CONFIDENCE_THRESHOLD:
        return card_low_conf(conf), ""
    return card_result(get_info(CLASS_MAP[top]), conf), ""

# ──────────────────────────────────────────────
# Cards — use rgba backgrounds so text contrast
# is always guaranteed regardless of theme
# Text is always white-on-dark or very-dark-on-light
# We use DARK CARD backgrounds so white text always works
# ──────────────────────────────────────────────

def card_waiting():
    return """
<div style="background:#1e3a1e;border:2px solid #4caf50;border-radius:16px;
            padding:60px 24px;text-align:center;">
    <div style="font-size:56px;margin-bottom:16px;">🌱</div>
    <p style="color:#ffffff;font-size:16px;font-weight:700;margin:0 0 8px;
              font-family:Arial,sans-serif;">Awaiting Diagnosis</p>
    <p style="color:#a5d6a7;font-size:13px;margin:0;font-family:Arial,sans-serif;">
        Upload a leaf photo and click <b style="color:#69f0ae;">Diagnose Leaf</b>
    </p>
</div>"""

def card_error(error):
    return f"""
<div style="background:#3e0000;border:2px solid #f44336;border-radius:14px;
            padding:22px;font-family:Arial,sans-serif;">
    <p style="color:#ff8a80;font-weight:700;font-size:15px;margin:0 0 10px;">
        ⚠️ Model Not Loaded
    </p>
    <p style="color:#ffcdd2;font-size:13px;margin:0;">{error}</p>
    <p style="color:#ef9a9a;font-size:13px;margin-top:10px;">
        Run <code style="background:#7f0000;padding:2px 8px;border-radius:4px;
        color:#ff8a80;">python train.py</code> first.
    </p>
</div>"""

def card_low_conf(conf):
    return f"""
<div style="background:#3e2c00;border:2px solid #ffb300;border-radius:14px;padding:28px;
            font-family:Arial,sans-serif;">
    <div style="text-align:center;margin-bottom:18px;">
        <div style="font-size:48px;margin-bottom:10px;">🔁</div>
        <p style="color:#ffe082;font-size:19px;font-weight:700;margin:0 0 8px;">
            Photo Not Clear Enough
        </p>
        <p style="color:#ffecb3;font-size:14px;line-height:1.6;margin:0;">
            Confidence was only <b style="color:#ffca28;">{conf:.1f}%</b>
            — needs at least <b style="color:#ffca28;">70%</b> for a reliable result.
        </p>
    </div>
    <div style="background:#2e1f00;border:1px solid #ffb300;border-radius:10px;padding:16px;">
        <p style="color:#ffe082;font-weight:700;font-size:13px;margin:0 0 10px;
                  text-transform:uppercase;letter-spacing:0.5px;">
            📸 Tips for a Better Photo
        </p>
        <ul style="margin:0;padding-left:20px;">
            <li style="color:#fff8e1;font-size:13px;line-height:2.2;font-family:Arial,sans-serif;">Use <b style="color:#ffca28;">one leaf only</b> — remove background clutter</li>
            <li style="color:#fff8e1;font-size:13px;line-height:2.2;font-family:Arial,sans-serif;">Shoot in <b style="color:#ffca28;">natural daylight</b>, not under a bulb or flash</li>
            <li style="color:#fff8e1;font-size:13px;line-height:2.2;font-family:Arial,sans-serif;">Hold camera <b style="color:#ffca28;">20-30 cm</b> above the leaf, keep it steady</li>
            <li style="color:#fff8e1;font-size:13px;line-height:2.2;font-family:Arial,sans-serif;">Ensure the <b style="color:#ffca28;">leaf fills most of the frame</b></li>
            <li style="color:#fff8e1;font-size:13px;line-height:2.2;font-family:Arial,sans-serif;">Use a <b style="color:#ffca28;">plain white background</b> behind the leaf</li>
            <li style="color:#fff8e1;font-size:13px;line-height:2.2;font-family:Arial,sans-serif;">Wipe your <b style="color:#ffca28;">camera lens</b> before shooting</li>
        </ul>
    </div>
</div>"""

def card_result(info, conf):
    return f"""
<div style="background:#1a2e1a;border:2px solid {info['border']};border-radius:16px;
            padding:24px;font-family:Arial,sans-serif;">

    <div style="display:flex;align-items:flex-start;gap:14px;margin-bottom:16px;">
        <span style="font-size:42px;line-height:1;flex-shrink:0;">{info['emoji']}</span>
        <div>
            <p style="color:#ffffff;font-size:20px;font-weight:700;margin:0 0 10px;">
                {info['display']}
            </p>
            <div style="display:flex;gap:8px;flex-wrap:wrap;">
                <span style="background:{info['border']}33;border:1px solid {info['border']};
                             color:#ffffff;padding:3px 14px;border-radius:20px;
                             font-size:12px;font-weight:700;">
                    Severity: {info['severity']}
                </span>
                <span style="background:#1b5e2033;border:1px solid #4caf50;
                             color:#a5d6a7;padding:3px 14px;border-radius:20px;
                             font-size:12px;font-weight:700;">
                    Confidence: {conf:.1f}%
                </span>
            </div>
        </div>
    </div>

    <div style="background:#0d1a0d;border-radius:8px;height:8px;
                margin-bottom:20px;overflow:hidden;">
        <div style="width:{conf:.0f}%;height:100%;background:{info['bar']};
                    border-radius:8px;"></div>
    </div>

    <div style="background:#0d2b0d;border-left:4px solid #4caf50;
                border-radius:0 10px 10px 0;padding:14px 16px;margin-bottom:16px;">
        <p style="color:#69f0ae;font-size:11px;font-weight:700;margin:0 0 6px;
                  letter-spacing:0.8px;text-transform:uppercase;">
            💊 Recommendation
        </p>
        <p style="color:#e8f5e9;font-size:14px;line-height:1.8;margin:0;">
            {info['remedy']}
        </p>
    </div>

    <div style="background:#0a1f0a;border:1px solid #2e4a2e;border-radius:10px;
                padding:14px 16px;margin-bottom:18px;">
        <p style="color:#69f0ae;font-size:11px;font-weight:700;margin:0 0 8px;
                  letter-spacing:0.8px;text-transform:uppercase;">
            📋 What To Do Next
        </p>
        <ol style="margin:0;padding-left:18px;">
            <li style="color:#c8e6c9;font-size:13px;line-height:2.2;">Follow the recommendation above — act quickly for High severity.</li>
            <li style="color:#c8e6c9;font-size:13px;line-height:2.2;">Click <b style="color:#69f0ae;">Learn More</b> to understand this disease in depth.</li>
            <li style="color:#c8e6c9;font-size:13px;line-height:2.2;">Click <b style="color:#69f0ae;">Prevention Guide</b> to protect your healthy plants.</li>
            <li style="color:#c8e6c9;font-size:13px;line-height:2.2;">When in doubt, consult a local agronomist with this diagnosis.</li>
        </ol>
    </div>

    <div style="display:flex;gap:10px;flex-wrap:wrap;">
        <a href="{info['learn_url']}" target="_blank"
           style="flex:1;min-width:130px;display:block;background:#2e7d32;
                  color:#ffffff;text-decoration:none;padding:11px 14px;
                  border-radius:10px;text-align:center;font-size:13px;font-weight:700;">
            📖 Learn More
        </a>
        <a href="{info['prev_url']}" target="_blank"
           style="flex:1;min-width:130px;display:block;background:#1b5e20;
                  color:#a5d6a7;text-decoration:none;padding:11px 14px;
                  border-radius:10px;text-align:center;font-size:13px;font-weight:700;
                  border:1.5px solid #4caf50;">
            🛡️ Prevention Guide
        </a>
    </div>
</div>"""

# ──────────────────────────────────────────────
# Build UI
# ──────────────────────────────────────────────
def build_ui():
    import gradio as gr

    with gr.Blocks(title="🌿 Plant Disease Detector") as demo:

        # HEADER
        gr.HTML("""
        <div style="background:linear-gradient(135deg,#1a5c1e,#2e7d32,#4caf50);
                    border-radius:18px;padding:30px 36px;margin-bottom:20px;text-align:center;
                    box-shadow:0 6px 24px rgba(46,125,50,0.4);">
            <div style="font-size:46px;margin-bottom:8px;">🌿</div>
            <p style="color:#ffffff;font-size:1.9rem;font-weight:900;margin:0 0 6px;
                      font-family:Georgia,serif;text-shadow:0 2px 8px rgba(0,0,0,0.3);">
                Plant Disease Detector
            </p>
            <p style="color:#c8e6c9;font-size:14px;margin:0;font-family:Arial,sans-serif;">
                🫑 Bell Pepper &nbsp;·&nbsp; 🍅 Tomato &nbsp;·&nbsp; 🥔 Potato
                &nbsp;|&nbsp; Powered by PlantVillage AI
            </p>
        </div>
        """)

        with gr.Row():

            # LEFT
            with gr.Column(scale=1):
                gr.HTML("""
                <div style="background:#2e2000;border:2px solid #ffb300;
                            border-radius:14px;padding:18px 20px;margin-bottom:14px;">
                    <p style="color:#ffe082;font-size:13px;font-weight:700;margin:0 0 12px;
                               text-transform:uppercase;letter-spacing:0.8px;
                               font-family:Arial,sans-serif;">
                        📷 How to Photograph Your Leaf
                    </p>
                    <ol style="margin:0;padding-left:20px;">
                        <li style="color:#fff8e1;font-size:13.5px;line-height:2.3;font-family:Arial,sans-serif;">Pick <b style="color:#ffca28;">one single leaf</b> — diseased or healthy.</li>
                        <li style="color:#fff8e1;font-size:13.5px;line-height:2.3;font-family:Arial,sans-serif;">Lay it flat on a <b style="color:#ffca28;">plain white surface</b>.</li>
                        <li style="color:#fff8e1;font-size:13.5px;line-height:2.3;font-family:Arial,sans-serif;">Shoot in <b style="color:#ffca28;">natural daylight</b> — no flash.</li>
                        <li style="color:#fff8e1;font-size:13.5px;line-height:2.3;font-family:Arial,sans-serif;">Hold camera <b style="color:#ffca28;">20-30 cm</b> above the leaf.</li>
                        <li style="color:#fff8e1;font-size:13.5px;line-height:2.3;font-family:Arial,sans-serif;">Keep leaf <b style="color:#ffca28;">sharp, centred and filling the frame</b>.</li>
                        <li style="color:#fff8e1;font-size:13.5px;line-height:2.3;font-family:Arial,sans-serif;">Wipe your <b style="color:#ffca28;">camera lens</b> before shooting.</li>
                    </ol>
                </div>
                """)

                image_input = gr.Image(
                    type="pil",
                    label="Upload or drag your leaf photo here",
                    height=290,
                )
                submit_btn = gr.Button("🔍  Diagnose Leaf", variant="primary")

            # RIGHT
            with gr.Column(scale=1):
                result_html = gr.HTML(card_waiting())
                hidden_out  = gr.Textbox(visible=False)

        # FOOTER
        gr.HTML("""
        <div style="text-align:center;color:#888;font-size:12px;
                    padding:16px 0 6px;margin-top:16px;font-family:Arial,sans-serif;">
            🌾 Trained on PlantVillage &nbsp;·&nbsp;
            Bell Pepper, Tomato &amp; Potato &nbsp;·&nbsp;
            <em>Always confirm with a certified agronomist.</em>
        </div>
        """)

        submit_btn.click(
            fn=predict,
            inputs=[image_input],
            outputs=[result_html, hidden_out],
        )

    return demo

# ──────────────────────────────────────────────
# Entry point
# ──────────────────────────────────────────────
if __name__ == "__main__":
    try:
        import gradio
    except ImportError:
        print("Run:  pip install gradio")
        exit(1)

    # ── Paste your ngrok token here ──
    NGROK_TOKEN = "3Fcjflc4bywb1ZXjrY6ngwhCW1s_6gZP5xwVJgmzXZQ9nf7i9"
    # ─────────────────────────────────
    PORT = 7879

    try:
        from pyngrok import ngrok, conf
        if NGROK_TOKEN and NGROK_TOKEN != "PASTE_YOUR_TOKEN_HERE":
            conf.get_default().auth_token = NGROK_TOKEN
            for t in ngrok.get_tunnels():
                ngrok.disconnect(t.public_url)
            tunnel = ngrok.connect(PORT, "http")
            print("\n" + "═" * 55)
            print(f"  🌍  Public URL  →  {tunnel.public_url}")
            print(f"  💻  Local  URL  →  http://localhost:{PORT}")
            print("═" * 55 + "\n")
        else:
            print(f"\n  💻  Local URL → http://localhost:{PORT}\n")
    except ImportError:
        print(f"\n  💻  Local URL → http://localhost:{PORT}\n")

    build_ui().launch(
        server_name="0.0.0.0",
        server_port=PORT,
        show_error=True,
        quiet=True,
    )

t=2026-06-25T16:27:49+0530 lvl=warn msg="Stopping forwarder" name=http-7860-35dff8b5-3899-46ca-b7e8-2e2a822e0eda acceptErr="failed to accept connection: Listener closed"



═══════════════════════════════════════════════════════
  🌍  Public URL  →  https://dating-patriarch-embolism.ngrok-free.dev
  💻  Local  URL  →  http://localhost:7879
═══════════════════════════════════════════════════════



In [ ]:
3Fcjflc4bywb1ZXjrY6ngwhCW1s_6gZP5xwVJgmzXZQ9nf7i9

In [ ]:
li{
color:black;
}
strong{
color:green;
}

background: rgb(255 233 124);